# Prática 1.4 — Distribuições e TCL na prática: sensores industriais

**Curso:** Estatística e Reconhecimento de Padrões na Ciência de Dados · **Giselle Falcão Academy**

Bem-vindo(a) à primeira prática do curso! Você vai analisar leituras do **motor M-07** de uma planta industrial fictícia (dados didáticos com valores realistas, embutidos neste notebook) e depois **construir o Teorema Central do Limite com as próprias mãos**.

**Pergunta condutora:** *qual é a impressão digital estatística de um motor saudável — e como a incerteza da minha média diminui quando eu coleto mais dados?*

**Como usar este notebook:**
1. Execute as células **na ordem**, de cima para baixo (botão ▶ ou `Shift+Enter`).
2. Leia os textos entre as células — eles conectam o código com as aulas 1.1 a 1.3.
3. No final há um **desafio** com TODOs. Não pule: é nele que o aprendizado acontece.

> Dica: salve sua cópia em **Arquivo → Salvar uma cópia no Drive** antes de começar.

In [ ]:
# Importando as bibliotecas — todas já vêm instaladas no Google Colab.
import numpy as np                 # arrays e simulação
from scipy import stats            # distribuições, ajustes e QQ-plot
import matplotlib.pyplot as plt    # gráficos

# Semente fixa: os sorteios aleatórios ficam reprodutíveis
# (seus gráficos vão bater com o gabarito da aula).
np.random.seed(42)

print("Bibliotecas prontas! numpy", np.__version__)

## 1. O histograma é o retrato da distribuição

Abaixo estão **60 leituras horárias de temperatura (°C) do mancal do motor M-07**, coletadas com a máquina operando em condição saudável.

Lembra da aula 1.1: o histograma não é só um gráfico descritivo — é o **retrato empírico da distribuição**, a impressão digital estatística da máquina funcionando bem.

Antes de executar, faça seu palpite: a distribuição deve ser simétrica? Com uma moda só?

In [ ]:
# 60 leituras horárias de temperatura do mancal (°C) — dados embutidos.
temperaturas = np.array([
    61.8, 62.3, 61.2, 60.9, 62.7, 61.5, 63.1, 62.0, 61.7, 60.4,
    62.5, 61.9, 62.2, 63.4, 61.1, 60.8, 62.9, 61.6, 62.4, 61.3,
    60.6, 63.0, 62.1, 61.4, 62.6, 60.2, 61.8, 62.8, 63.6, 61.0,
    62.3, 61.7, 60.7, 62.0, 63.2, 61.5, 62.5, 61.2, 60.5, 62.7,
    61.9, 63.3, 62.2, 61.6, 60.9, 62.4, 61.3, 62.9, 61.1, 62.6,
    60.3, 61.8, 63.5, 62.1, 61.4, 62.8, 61.0, 62.3, 63.0, 61.7
])

print(f"n = {len(temperaturas)} leituras")
print(f"Média  : {temperaturas.mean():.2f} °C")
print(f"Desvio : {temperaturas.std(ddof=1):.2f} °C")

plt.figure(figsize=(7, 4))
plt.hist(temperaturas, bins=10, color="#6D28D9", edgecolor="white")  # roxo da Academy
plt.xlabel("Temperatura do mancal (°C)")
plt.ylabel("Número de leituras")
plt.title("Motor M-07 saudável — o retrato da distribuição")
plt.show()

## 2. Ajustando uma normal e conferindo no QQ-plot

O histograma parece um sino — mas "parece" não é diagnóstico. Vamos fazer duas coisas:

1. **Ajustar uma normal** aos dados com `stats.norm.fit`, que estima μ e σ, e sobrepor a curva ao histograma.
2. Construir o **QQ-plot** (aula 1.2): se os pontos abraçarem a reta — *inclusive nas pontas* — a normal é uma boa descrição.

É este teste que evita o erro caro da aula 1.1: projetar um alarme μ ± 3σ para um dado que não é normal.

In [ ]:
# Ajuste da normal: fit devolve os parâmetros que melhor descrevem os dados.
mu, sigma = stats.norm.fit(temperaturas)
print(f"Normal ajustada: mu = {mu:.2f} °C, sigma = {sigma:.2f} °C")

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

# Esquerda: histograma em densidade + curva da normal ajustada.
# density=True coloca histograma e curva na mesma escala!
eixos[0].hist(temperaturas, bins=10, density=True, color="#6D28D9",
              edgecolor="white", alpha=0.7)
x = np.linspace(temperaturas.min() - 1, temperaturas.max() + 1, 200)
eixos[0].plot(x, stats.norm.pdf(x, mu, sigma), color="#0D9488", lw=2.5)  # teal
eixos[0].set_xlabel("Temperatura (°C)")
eixos[0].set_ylabel("Densidade")
eixos[0].set_title("Histograma + normal ajustada")

# Direita: QQ-plot contra a normal teórica.
stats.probplot(temperaturas, dist="norm", plot=eixos[1])
eixos[1].set_title("QQ-plot — os pontos abraçam a reta?")

plt.tight_layout()
plt.show()

**Sua leitura (edite esta célula):** os pontos do QQ-plot seguem a reta, inclusive nas pontas? A temperatura do M-07 parece bem descrita por uma normal? *Escreva sua conclusão em 1–2 linhas aqui.*

---

## 3. O experimento das mil amostras: o TCL na sua tela

Agora o gráfico mais importante do módulo. Vamos pegar uma população **torta de propósito** — o tempo entre falhas de equipamentos, uma exponencial com média de 30 dias (leitura 1.2, ficha 5) — e executar o protocolo da aula 1.3:

> sorteie n valores → calcule a média → repita 2.000 vezes → faça o histograma **das médias**.

Se o Teorema Central do Limite estiver certo, o histograma das médias vai virar um sino conforme n cresce — mesmo a população continuando torta.

In [ ]:
media_populacao = 30          # média verdadeira do tempo entre falhas (dias)
populacao = stats.expon(scale=media_populacao)   # exponencial: bem assimétrica

tamanhos = [2, 10, 50]        # os três n do experimento
n_repeticoes = 2000           # quantas amostras sorteamos para cada n

fig, eixos = plt.subplots(1, 4, figsize=(15, 3.5))

# Painel 0: a população original, torta.
eixos[0].hist(populacao.rvs(5000), bins=40, color="#6D28D9", edgecolor="white")
eixos[0].set_title("População: exponencial (torta!)")
eixos[0].set_xlabel("Dias entre falhas")

# Painéis 1-3: histograma DAS MÉDIAS para cada n.
for eixo, n in zip(eixos[1:], tamanhos):
    medias = populacao.rvs((n_repeticoes, n)).mean(axis=1)  # 2000 médias de amostras de tamanho n
    eixo.hist(medias, bins=40, color="#0D9488", edgecolor="white")
    eixo.axvline(media_populacao, color="#6D28D9", ls="--", lw=2)
    eixo.set_title(f"Médias com n = {n}")
    eixo.set_xlabel("Média amostral (dias)")

plt.tight_layout()
plt.show()
print("A linha tracejada é a média verdadeira (30 dias).",
      "Repare: a população segue torta, mas as MÉDIAS viram um sino.")

## 4. Erro padrão e a lei do √n

A aula 1.3 prometeu: o desvio da distribuição das médias é o **erro padrão** σ/√n — e para cortá-lo pela metade é preciso **quadruplicar** os dados.

Vamos conferir os dois lados:
1. Comparar o desvio **empírico** das médias simuladas com o **teórico** σ/√n.
2. Usar o erro padrão para construir um intervalo de ~95% para a temperatura média do M-07 (x̄ ± 2·EP) — a semente do intervalo de confiança do módulo 2.

In [ ]:
sigma_pop = populacao.std()   # desvio da exponencial (= média, 30 dias)

print("Lei do raiz de n — desvio das médias simuladas vs. teoria (sigma/raiz(n)):")
for n in tamanhos:
    medias = populacao.rvs((n_repeticoes, n)).mean(axis=1)
    ep_empirico = medias.std(ddof=1)
    ep_teorico = sigma_pop / np.sqrt(n)
    print(f"  n = {n:3d} -> empírico {ep_empirico:5.2f} | teórico {ep_teorico:5.2f} dias")

# Agora o M-07: intervalo de ~95% para a temperatura média.
n = len(temperaturas)
media = temperaturas.mean()
ep = temperaturas.std(ddof=1) / np.sqrt(n)   # erro padrão da média

print(f"\nMotor M-07: média = {media:.2f} °C, erro padrão = {ep:.3f} °C")
print(f"Intervalo de ~95%: [{media - 2*ep:.2f}, {media + 2*ep:.2f}] °C")
print("Leitura: nossa melhor estimativa da temperatura média de operação",
      "do M-07 saudável, com a incerteza declarada. No módulo 2, essa frase",
      "ganha o nome e o rigor de 'intervalo de confiança'.")

## 5. 🏆 Desafio — o sensor de vibração do M-07

A equipe de manutenção enviou **40 leituras de vibração RMS (mm/s)** do mesmo motor. Sua missão é repetir o diagnóstico completo — agora sem código pronto.

Complete os TODOs na célula abaixo (remova o `#` e preencha as lacunas `____`):

1. **Histograma** da vibração + média e desvio impressos.
2. **QQ-plot** contra a normal teórica.
3. Responda na célula de texto final: a vibração parece normal? O que os pontos que fogem da reta **na ponta direita** sugerem sobre o M-07? Por que um alarme μ + 3σ seria arriscado aqui?

> Dica: os padrões são os mesmos das seções 1 e 2 — adaptar código que funciona é parte do ofício.

In [ ]:
# 40 leituras de vibração RMS (mm/s) do M-07 — dados embutidos.
vibracao = np.array([
    2.8, 2.5, 3.1, 2.9, 2.6, 3.0, 2.7, 3.2, 2.4, 2.9,
    3.1, 2.8, 2.6, 3.3, 2.7, 2.5, 3.0, 2.9, 2.8, 3.2,
    2.6, 3.1, 2.7, 2.9, 3.4, 2.8, 2.5, 3.0, 2.6, 3.2,
    2.9, 2.7, 4.1, 3.0, 2.8, 4.6, 3.1, 2.6, 5.2, 2.9
])

# === DESAFIO — complete os TODOs abaixo ===

# TODO 1: média, desvio e histograma da vibração
# print(f"Média : {vibracao.____():.2f} mm/s")
# print(f"Desvio: {vibracao.std(ddof=1):.2f} mm/s")
# plt.hist(____, bins=12, color="#6D28D9", edgecolor="white")
# plt.xlabel("Vibração RMS (mm/s)")
# plt.ylabel("Número de leituras")
# plt.title("Motor M-07 — vibração")
# plt.show()

# TODO 2: QQ-plot da vibração contra a normal
# stats.probplot(____, dist="norm", plot=plt)
# plt.title("QQ-plot da vibração — e agora?")
# plt.show()

**TODO 3 — seu diagnóstico (edite esta célula, até 4 linhas):**

*A vibração parece normal? O que a cauda direita (as leituras 4,1 / 4,6 / 5,2) sugere sobre o estado do M-07? Por que um alarme fixado em μ + 3σ seria arriscado aqui?*

---

## Parabéns! 🎉

Você fez o diagnóstico distribucional completo de um sensor real-de-exemplo, viu o Teorema Central do Limite emergir na tela e quantificou a incerteza de uma média com o erro padrão — o alicerce de todo o resto do curso.

**Checklist de conclusão (marque na plataforma):**

- [ ] Executei todas as células na ordem, sem erro.
- [ ] Interpretei o QQ-plot da temperatura (seção 2).
- [ ] Identifiquei em qual n o sino "aparece" na simulação do TCL.
- [ ] Confirmei a lei do √n na tabela da seção 4.
- [ ] Completei os TODOs do desafio e escrevi o diagnóstico da vibração.
- [ ] Enviei o link compartilhável do notebook na plataforma (**Compartilhar → qualquer pessoa com o link → Leitor**).

**No Módulo 2**, o intervalo "caseiro" x̄ ± 2·EP vira intervalo de confiança de verdade, e você aprende a comparar dois grupos com testes de hipótese e bootstrap. Te vejo lá! — *Giselle*